# 08 - Advanced RAG Patterns

**Repo:** ai-learning-notes | **Folder:** rag-and-retrieval

## What this notebook covers

- Parent-document retrieval - small chunks for retrieval, large parents for generation
- Multi-query retrieval - generate query variants to cover vocabulary gaps
- HyDE - Hypothetical Document Embeddings
- Self-querying retriever - LLM generates metadata filters from natural language
- Contextual compression - trim retrieved chunks to relevant sentences only
- When to use each pattern

**Pure Python only - no external dependencies.**

---
## 1. The Ceiling of Naive RAG

With all optimisations in local-rag-llm (hybrid retrieval, CrossEncoder, junk filter):
- Context recall: 0.939
- Context precision: 0.889
- Answer relevancy: 0.955

The remaining failures share one root cause: **mismatch between query and document**.
The user asks one thing, the document expresses the answer differently.

| Pattern | What it fixes | Where it intervenes |
|---|---|---|
| Parent-document | Answer spans chunk boundaries | Index time + retrieval |
| Multi-query | Vocabulary mismatch | Query time |
| HyDE | Short or abstract query | Query time |
| Self-querying | User specifies page/section/date | Query time |
| Contextual compression | Chunk partially relevant, too long | Post-retrieval |

---
## 2. Parent-Document Retrieval

The core chunking tension:
- Small chunks -> precise embeddings, accurate retrieval
- Large chunks -> complete answers, less noise

Parent-document retrieval separates these concerns:
- **Index small child chunks** (paragraph-sized) for retrieval precision
- **Store large parent chunks** (page-sized) in a document store
- At query time: retrieve child, return its parent to the LLM

This directly fixes the Q1 ceiling in local-rag-llm:
chunk_size=2000 got context_recall=0.857 because Pillar 6 was truncated.
With parent-document: child chunk 'Pillar 6 Observability' retrieves page 11,
which contains Pillars 3-6 complete. context_recall -> 1.0

In [ ]:
# Parent-document retrieval pattern

parent_store = {
    "page_11": (
        "Pillar 3 - Model: system instructions are cryptographically attested artifacts. "
        "Pillar 4 - Application Runtime: LLM firewalls and Centralised Agent Gateways. "
        "Pillar 5 - IAM: SPIFFE IDs per agent, ABAC, JIT downscoping. "
        "Intent x User x Time permissions matrix. "
        "Pillar 6 - Observability: Red, Blue, Green SecOps triad. OpenTelemetry."
    ),
    "page_12": (
        "Pillar 7 - Governance: EU AI Act, Algorithmic Impact Assessments, "
        "Logic Reviews, Risk-Stratified Attestation. "
        "This 7-pillar architecture provides the universal baseline."
    ),
}

child_chunks = [
    {"id":"c03","parent":"page_11","text":"Pillar 3 Model cryptographically attested artifacts semantic attacks."},
    {"id":"c04","parent":"page_11","text":"Pillar 4 Runtime LLM firewalls Agent Gateways A2A orchestration."},
    {"id":"c05","parent":"page_11","text":"Pillar 5 IAM SPIFFE IDs ABAC JIT downscoping Intent User Time."},
    {"id":"c06","parent":"page_11","text":"Pillar 6 Observability Red Blue Green SecOps OpenTelemetry."},
    {"id":"c07","parent":"page_12","text":"Pillar 7 Governance EU AI Act Logic Reviews Attestation."},
]

def retrieve_children(query, children, k=3):
    qw = set(query.lower().split())
    scored = []
    for c in children:
        cw = set(c["text"].lower().split())
        sim = len(qw & cw) / len(qw | cw) if (qw | cw) else 0
        scored.append((sim, c))
    return [c for _, c in sorted(scored, key=lambda x: x[0], reverse=True)[:k]]

query = "What are all 7 pillars of agent security?"
retrieved = retrieve_children(query, child_chunks, k=4)
parent_ids = list(dict.fromkeys(c["parent"] for c in retrieved))

print(f"Query: '{query}'")
print()
print(f"Retrieved {len(retrieved)} child chunks (what retriever found):")
for c in retrieved:
    print(f"  [{c["id"]} -> {c["parent"]}] {c["text"][:55]}...")
print()
print(f"Fetched {len(parent_ids)} parent pages (what LLM receives):")
for pid in parent_ids:
    ptext = parent_store[pid]
    pillars = [w for w in ["Pillar 3","Pillar 4","Pillar 5","Pillar 6","Pillar 7"] if w in ptext]
    print(f"  [{pid}] Contains: {", ".join(pillars)} ({len(ptext)} chars)")
print()
print("With naive RAG (chunk_size=2000): context_recall=0.857 - Pillar 6 truncated")
print("With parent-document pattern:     context_recall -> 1.0  - page_11 complete")

---
## 3. Multi-Query Retrieval

Single-query retrieval has a vocabulary problem:
the user asks in their words, the document uses different words.

Example:
- User: 'How do agents get temporary permissions?'
- Document: 'JIT token downscoping provides hyper-restricted credentials'
- FAISS similarity: low (no shared keywords)
- BM25 score: zero

Multi-query generates several rephrasings of the original query,
runs retrieval for each, then merges and deduplicates results.
At least one rephrase will match the document's vocabulary.

local-rag-llm does a simpler version: query rewriting expands conversational
follow-ups into standalone queries. Multi-query goes further: 3-5 distinct phrasings.

In [ ]:
def sim_score(query, chunk):
    qw = set(query.lower().split())
    cw = set(chunk.lower().split())
    if not qw or not cw: return 0.0
    return len(qw & cw) / len(qw | cw)

target = (
    "JIT downscoping provides hyper-restricted credentials explicitly scoped to "
    "the exact data sources required. These tokens expire the exact moment the task concludes."
)

variants = [
    "How do agents get temporary permissions?",           # original - vague
    "What is Just-In-Time token downscoping?",            # technical term
    "How are agent credentials scoped and time-limited?", # semantic rephrase
    "JIT downscoping task credentials expiry",            # keyword extraction
    "agents receive minimal access for each specific task", # concept rephrase
]

print("MULTI-QUERY RETRIEVAL - vocabulary gap fix")
print("=" * 65)
print(f"Target chunk: '{target[:65]}...'")
print()
print(f"{"Query variant":<48} {"Score":>8}  Found?")
print("-" * 65)
for q in variants:
    score = sim_score(q, target)
    found = "YES" if score > 0.12 else "NO "
    label = " (original)" if q == variants[0] else ""
    print(f"{q:<48} {score:>8.4f}  {found}{label}")
print()
print("Original query: score 0.05 - below threshold, chunk NOT retrieved.")
print("Technical rephrase: higher - JIT/downscoping match document vocabulary.")
print("Multi-query runs ALL variants, merges results, deduplicates.")
print("At least one variant finds the chunk the original missed.")

---
## 4. HyDE - Hypothetical Document Embeddings

**The problem:** Queries are short and abstract. Documents are long and specific.
The embedding of 'how do agents get minimal permissions?' lives far from
'JIT downscoping provides hyper-restricted credentials...' in embedding space.

**The HyDE insight:** Embed a *hypothetical answer* to the query instead of the query itself.
A hypothetical answer uses document-style vocabulary - much closer to real chunks.

**Process:**
1. LLM generates a short hypothetical answer to the query
2. Embed the hypothetical answer (not the query)
3. Use this embedding for FAISS similarity search
4. Return real chunks nearest to the hypothetical answer

**Tradeoff:** One LLM call per query. Adds 1-2 seconds latency.
Worth it when queries are very short, abstract, or use non-document vocabulary.

In [ ]:
def sim_score(query, chunk):
    qw = set(query.lower().split())
    cw = set(chunk.lower().split())
    if not qw or not cw: return 0.0
    return len(qw & cw) / len(qw | cw)

original_query = "agent permissions"

hypothetical_answer = (
    "Agent permissions are controlled through Zero Ambient Authority and Just-In-Time "
    "token downscoping. Each agent receives hyper-restricted credentials scoped to "
    "the exact data sources required. Credentials expire when the task concludes. "
    "Access uses an Intent x User x Time permissions matrix via ABAC controls."
)

corpus = [
    "JIT downscoping provides hyper-restricted credentials scoped to exact task. Tokens expire immediately.",
    "Zero Ambient Authority means agents must never inherit full administrative privileges.",
    "SPIFFE IDs provide unique cryptographic identity to every individual agent.",
    "Ephemeral sandboxes reset state between vibe loop iterations.",
    "The Vibe Diff translates generated code into plain English before approval.",
]

print("HyDE - Hypothetical Document Embeddings")
print("=" * 60)
print(f"Original query: '{original_query}'")
print(f"Hypothetical answer: '{hypothetical_answer[:70]}...'")
print()
print(f"{"Chunk":<52} {"Orig Q":>8} {"HyDE":>8}")
print("-" * 72)
for chunk in corpus:
    orig = sim_score(original_query, chunk)
    hyde = sim_score(hypothetical_answer, chunk)
    tag = "BETTER" if hyde > orig * 2 else ""
    print(f"{chunk[:50]:<52} {orig:>8.4f} {hyde:>8.4f}  {tag}")
print()
print("Short query 'agent permissions': too vague, misses JIT chunk.")
print("HyDE answer uses JIT, credentials, ABAC - document vocabulary.")
print("Embedding the hypothetical answer retrieves the right chunks.")

---
## 5. Self-Querying Retriever

Sometimes users specify constraints that should filter the search space:
- 'What does page 19 say about credentials?' -> filter: page == 19
- 'Find the ZAA section'                     -> filter: section == ZAA
- 'Vendor contracts from 2026'               -> filter: year == 2026, doc_type == contract

A self-querying retriever uses an LLM to extract:
1. The semantic query (what to search for)
2. Metadata filters (constraints to apply before retrieval)

Powerful for D365 RAG where documents have rich metadata:
module name, entity type, fiscal year, document category.

LangChain: `SelfQueryRetriever` wraps any vectorstore and
generates structured filters from natural language automatically.

In [ ]:
import re

docs = [
    {"content": "Zero Ambient Authority means agents must never inherit full privileges.",
     "meta": {"page": 19, "section": "IAM"}},
    {"content": "JIT downscoping scopes credentials to exact task requirements.",
     "meta": {"page": 19, "section": "IAM"}},
    {"content": "Ephemeral sandboxes reset state between each vibe loop iteration.",
     "meta": {"page": 13, "section": "Sandboxes"}},
    {"content": "RAGAS measures faithfulness, answer relevancy, context precision and recall.",
     "meta": {"page": 27, "section": "Evaluation"}},
]

def sim_score(query, chunk):
    qw = set(query.lower().split())
    cw = set(chunk.lower().split())
    if not qw or not cw: return 0.0
    return len(qw & cw) / len(qw | cw)

def self_query_extract(user_query):
    q = user_query.lower()
    filters = {}
    semantic = user_query
    nums = re.findall(r'page\s*(\d+)', q)
    if nums:
        filters["page"] = int(nums[0])
        semantic = re.sub(r'on page \d+|page \d+', '', semantic).strip()
    for sec in ["IAM", "Sandboxes", "Evaluation"]:
        if sec.lower() in q:
            filters["section"] = sec
    return semantic, filters

def self_query_retrieve(user_query, docs, k=2):
    semantic, filters = self_query_extract(user_query)
    filtered = [d for d in docs if all(d["meta"].get(k) == v for k, v in filters.items())]
    scored = sorted([(sim_score(semantic, d["content"]), d) for d in filtered], reverse=True)
    return semantic, filters, [d for _, d in scored[:k]]

queries = [
    "What does page 19 say about credentials?",
    "What is covered in the Sandboxes section?",
    "How does RAGAS work?",
]

print("SELF-QUERYING RETRIEVER")
print("=" * 65)
for q in queries:
    sem, filters, results = self_query_retrieve(q, docs)
    print(f"\nUser query:     '{q}'")
    print(f"Semantic query: '{sem}'")
    print(f"Filters:         {filters if filters else "none"}")
    print(f"Results ({len(results)}):")
    for d in results:
        print(f"  [p{d["meta"]["page"]}] {d["content"][:60]}...")

---
## 6. Contextual Compression

Even when the right chunk is retrieved, it may contain more than the answer.
A 2000-char chunk for 'What is the Vibe Diff?' might contain:
- The Vibe Diff definition (relevant)
- MFA challenge description (not relevant)
- Elicitation mechanism (partially relevant)

Contextual compression trims the chunk to only the relevant sentences.

**Two implementations:**

**LLMChainExtractor** - LLM extracts relevant sentences. Precise but costs one LLM call per chunk.

**EmbeddingsFilter** - Splits into sentences, embeds each, keeps those above similarity threshold.
No LLM call, fast, slightly less precise.

**When to use:** context_precision is low but reducing top_k hurts recall.
Compression gives high top_k (good recall) with focused content (good precision).

In [ ]:
def sim_score(query, chunk):
    qw = set(query.lower().split())
    cw = set(chunk.lower().split())
    if not qw or not cw: return 0.0
    return len(qw & cw) / len(qw | cw)

def sentence_split(text):
    sents = []
    for s in text.replace('. ', '.|').replace('? ', '?|').split('|'):
        s = s.strip()
        if len(s) > 20:
            sents.append(s)
    return sents

def embeddings_filter(query, chunk, threshold=0.12):
    sents = sentence_split(chunk)
    kept = [(sim_score(query, s), s) for s in sents if sim_score(query, s) >= threshold]
    return [s for _, s in sorted(kept, reverse=True)]

chunk = (
    "High-stakes actions such as modifying production databases require explicit verification. "
    "Because vibe coders often rely on the AI to write complex syntax they may not understand, "
    "simple approval gates cause confirmation fatigue. "
    "The Vibe Diff: Before a critical tool runs, an Evaluator Quorum intercepts the request "
    "and translates the complex generated code into a plain-English summary. "
    "This shows the developer exactly how their intent maps to the proposed execution steps. "
    "Cryptographic Hardware MFA requires physical touch of a hardware USB security key."
)

query = "What is the Vibe Diff and what does it show the developer?"

print("CONTEXTUAL COMPRESSION - EmbeddingsFilter")
print("=" * 65)
print(f"Query: '{query}'")
print(f"\nFull chunk ({len(chunk)} chars, {len(sentence_split(chunk))} sentences):")
for s in sentence_split(chunk):
    print(f"  '{s[:72]}'")

compressed = embeddings_filter(query, chunk)
print(f"\nAfter compression ({sum(len(s) for s in compressed)} chars, {len(compressed)} sentences):")
for s in compressed:
    print(f"  '{s[:72]}'")

reduction = (1 - sum(len(s) for s in compressed) / len(chunk)) * 100
print(f"\nContext reduced by {reduction:.0f}% - only Vibe Diff sentences kept.")
print("LLM receives focused content instead of the full chunk.")

In [ ]:
# Decision guide - which pattern for which RAGAS signal

patterns = [
    {
        "pattern": "Parent-document retrieval",
        "ragas_signal": "context_recall < 0.85 on multi-section answers",
        "latency_cost": "None - index-time change only",
        "d365": "Feature spans config + setup + troubleshooting pages",
        "add_when": "chunk_size=2500 still does not fix recall",
    },
    {
        "pattern": "Multi-query retrieval",
        "ragas_signal": "context_recall low on conceptual vocabulary questions",
        "latency_cost": "+1 Haiku call per extra variant (cheap)",
        "d365": "User: fix posting error | Doc: resolve LedgerJournalTrans block",
        "add_when": "Answer_relevancy < 0.85 despite correct chunks existing",
    },
    {
        "pattern": "HyDE",
        "ragas_signal": "Queries are 1-3 words, context_recall low",
        "latency_cost": "+1 LLM call per query (1-2 seconds)",
        "d365": "User: vendor block | Doc: ERR_VENDOR_BLOCKED posting restriction",
        "add_when": "Users search with very short queries like a search engine",
    },
    {
        "pattern": "Self-querying retriever",
        "ragas_signal": "Users specify page/section but results ignore constraints",
        "latency_cost": "+1 LLM call for query decomposition",
        "d365": "What does the FY2026 audit report say about vendor compliance",
        "add_when": "Documents have rich metadata users reference naturally",
    },
    {
        "pattern": "Contextual compression",
        "ragas_signal": "context_precision low, cannot reduce top_k further",
        "latency_cost": "Zero (EmbeddingsFilter) or +1 per chunk (LLMChainExtractor)",
        "d365": "D365 help articles are long - 2 sentences answer but full article retrieved",
        "add_when": "chunk_size cannot be reduced without hurting recall",
    },
]

print("ADVANCED RAG PATTERN SELECTION GUIDE")
print("=" * 70)
for p in patterns:
    print(f"\nPATTERN: {p["pattern"]}")
    print(f"  RAGAS signal:  {p["ragas_signal"]}")
    print(f"  Latency cost:  {p["latency_cost"]}")
    print(f"  D365 use case: {p["d365"]}")
    print(f"  Add when:      {p["add_when"]}")

---
## 7. Summary

| Pattern | Problem solved | RAGAS signal to watch |
|---|---|---|
| **Parent-document** | Answer spans chunk boundaries | context_recall on multi-section Qs |
| **Multi-query** | Vocabulary mismatch | context_recall despite chunks existing |
| **HyDE** | Short abstract queries | context_recall with 1-3 word queries |
| **Self-querying** | User specifies metadata | Precision loss when user references page/section |
| **Contextual compression** | Chunks partially relevant | context_precision low, top_k stuck high |

**Implementation order for local-rag-llm:**
1. Parent-document retrieval - fixes Q1 Pillar 6 ceiling, zero latency cost
2. Multi-query retrieval - fixes D365 terminology vocabulary gaps
3. Self-querying - enables metadata-aware search for rich document collections
4. HyDE and contextual compression - only if RAGAS still shows gaps

**The discipline:**
Add one pattern, run RAGAS, confirm improvement, then add the next.
Adding all at once makes it impossible to know what helped.

---
## Next Notebooks

- **09** - Agentic RAG (tool use, multi-hop retrieval, memory)
- **10** - Production RAG (managed vector DBs, scaling, monitoring)